# Field validation — `frontal_structure` (SURF pipeline)

End-to-end validation of every calculated field in the
**frontal_structure** surface subset: the notebook RUNs the pipeline,
LOADs its own output, and validates each field with dependency-chain
maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `frontal_structure` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Re-run this notebook after any code change to the fields below to
re-validate the pipeline end to end.

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "frontal_structure"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = (list(defn["model_data_feature_channels"])
            + list(defn["compute_features_channels"]))

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `frontal_structure`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`; no raw
model channels in this subset — every channel is computed):

`gradb2`, `gradsalt2`, `gradtheta2`, `gradeta2`, `gradrho2`,
`turner_angle`, `density`, `buoyancy`

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert list(reader.channel_names) == CHANNELS, (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel list matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| density (σ₀) | kg m⁻³ | σ₀ = ρ_JMD95(S, Θ, p=0) − 1000 | Theta, Salt → rho_theta | `calculate_fields.potential_density_anomaly` |
| buoyancy (b) | m s⁻² | b = g·σ₀/ρ₀ | density (← Theta, Salt) | `calculate_fields.buoyancy_of_field` |
| gradtheta2 | °C² m⁻² | \|∇Θ\|² = Θₓ² + Θᵧ² | Theta; dxC, dyC, CS, SN | `calculate_fields.grad_theta2` → `native_gradient` |
| gradsalt2 | psu² m⁻² | \|∇S\|² | Salt; grid metrics | `calculate_fields.grad_salt2` → `native_gradient` |
| gradeta2 | m² m⁻² | \|∇η\|² | Eta; grid metrics | `calculate_fields.grad_eta2` → `native_gradient` |
| gradrho2 | kg² m⁻⁸ | \|∇ρθ\|² | rho_theta (← Theta, Salt); grid metrics | `calculate_fields.grad_rho2` → `native_gradient` |
| gradb2 | s⁻⁴ | \|∇b\|² | buoyancy (← density ← Theta, Salt); grid metrics | `calculate_fields.grad_b2` → `native_gradient` |
| turner_angle | ° | Tu = arctan[ρ₀(β²\|∇S\|² − α²\|∇Θ\|²) / (−\|∇ρ\|²/ρ₀)] | gradtheta2, gradsalt2, gradrho2 | `calculate_fields.turner_angle` |

Non-channel intermediate (computed live in this notebook, plotted in
the dependency columns):

| INTERMEDIATE | UNITS | EQUATION | DEPENDENCIES | LOCATION |
|---|---|---|---|---|
| rho_theta | kg m⁻³ | ρ_JMD95(S, Θ, p=0) | Theta, Salt | `calculate_fields.potential_density` |

Processing operations in this subset's pipeline: land masking
(`hFacC == 0`, NaN); native-grid tracer differentiation with CS/SN
rotation (leaves a NaN halo rim at face edges); **no** staggered→tracer
interpolation (all inputs are tracer-point scalars); face→lat-lon
stitching; global row downsampled `[::12, ::12]`.  Conventions:
σ₀ = ρ − 1000 (anomaly); b is anomaly-based (g·σ₀/ρ₀).

### Raw inputs & live intermediates

The store contains only the 8 output channels.  Raw inputs (Theta,
Salt, Eta) and the non-channel intermediate (rho_theta) are computed
LIVE from the same OSN snapshot with the same code, then stitched to
the rect grid — guaranteeing the plotted dependencies match the code
being validated.

In [ ]:
# Live raw + intermediate fields (same loaders as the pipeline).
import dbof.preprocessing.calculate_fields as calculate_fields
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

RAW_VARS = ["Theta", "Salt", "Eta"]      # raw inputs of this subset
LIVE = RAW_VARS + ["rho_theta"]          # + non-channel intermediate

# Grid + snapshot exactly as generate-global does for SURF.
ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, RAW_VARS,
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}  (store iteration {reader.iteration})")

# Assign the live fields onto the face dataset and stitch to rect.
ds_conv = ds_raw.assign({
    "Theta": ds_merge["Theta"],
    "Salt": ds_merge["Salt"],
    "Eta": ds_merge["Eta"],
    "rho_theta": calculate_fields.potential_density(ds_merge),
})[LIVE]
mask = {"_land_mask": (ds_merge.hFacC == 0)}
live_chw = stitch_and_mask(ds_conv, LIVE, mask)   # (4, H, W)
print(f"live fields stitched: {LIVE}, shape {live_chw.shape}")

In [ ]:
# Slice every field (live + store) to the four validation domains.
# Full-res global arrays are released immediately after slicing.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps

CMAP_CFG, DIVERGING = load_field_cmaps()

# The five ∝-squared gradient channels: log colour scale (maps) and
# log10-x binning (PDFs).
GRAD2 = {"gradb2", "gradsalt2", "gradtheta2", "gradeta2", "gradrho2"}

# region_arrays[field][region] = (x, y, arr)
region_arrays = {}
for k, ch in enumerate(LIVE):
    region_arrays[ch] = regions.select_all_regions(live_chw[k], XC, YC)
del live_chw                       # free ~4 full-res arrays
for ch in CHANNELS:
    arr = reader.get_channel_snapshot(ch)
    region_arrays[ch] = regions.select_all_regions(arr, XC, YC)
    del arr                        # keep peak memory at 1 full array

for ch in region_arrays:
    x, y, sub = region_arrays[ch]["gulf_stream"]
    print(f"{ch:14s} gulf_stream {sub.shape}  "
          f"min {np.nanmin(sub):.3g}  max {np.nanmax(sub):.3g}")

## Section 5 — Per-field validation

For each field, in dependency order (dependencies first — σ₀ and b are
the subset's own output channels and are validated here; sibling
notebooks reference these sections rather than re-validating):

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → intermediate → final; option (a) of the plan — each field's
  chain reads standalone), rows = domains A–D.  One shared colour
  scale per column.
- **Figure 2 — PDFs**: same grid.  Conventions: probability density;
  land + halo-rim NaNs removed; log10-x for ∝-squared fields; bins
  shared per field across the four domains.
- **Figure 3 — literature comparison**: (a) our generated data,
  (b) `.png` from literature.  Figure type will be specified
  explicitly per variable (plan §7); until then the default renderer
  is a Gulf Stream map and the right panel is a placeholder.
  Drop reference images in
  `notebooks/notebooks_field_validation/literature/frontal_structure/
  {field}.png`.

In [ ]:
# Section 5 helpers: one call per figure, shared by all 8 fields.
from pathlib import Path

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import pipeline_map_grid
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

LIT_DIR = Path("../literature/frontal_structure")

# Full dependency chain per field (columns of Figures 1–2).
CHAINS = {
    "density":      ["Theta", "Salt", "rho_theta", "density"],
    "buoyancy":     ["Theta", "Salt", "rho_theta", "density",
                     "buoyancy"],
    "gradtheta2":   ["Theta", "gradtheta2"],
    "gradsalt2":    ["Salt", "gradsalt2"],
    "gradeta2":     ["Eta", "gradeta2"],
    "gradrho2":     ["Theta", "Salt", "rho_theta", "gradrho2"],
    "gradb2":       ["Theta", "Salt", "density", "buoyancy", "gradb2"],
    "turner_angle": ["Theta", "Salt", "gradtheta2", "gradsalt2",
                     "gradrho2", "turner_angle"],
}

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; log10-x for "
            "\u221d-squared; shared bins across domains")


def figure1_maps(field):
    """Figure 1: dependency-chain map grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    pipeline_map_grid(
        CHAINS[field], region_arrays, CMAP_CFG,
        diverging_cmaps=DIVERGING, log_scale_channels=GRAD2,
        suptitle=f"Figure 1 \u2014 {field}: pipeline maps",
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: dependency-chain PDF grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    pipeline_pdf_grid(
        CHAINS[field], region_arrays, CMAP_CFG, log10_fields=GRAD2,
        suptitle=f"Figure 2 \u2014 {field}: {PDF_NOTE}",
    )
    plt.show()


def figure3_literature(field, caption=None):
    """Figure 3: our data vs literature .png for one field.

    Default renderer (until a figure type is specified per variable):
    Gulf Stream regional map of the final field.
    Inputs: field (str); caption (str or None) — discussion text.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    def _render(ax):
        x, y, arr = region_arrays[field]["gulf_stream"]
        im, label = plot_global_field(
            ax, x, y, arr, field, CMAP_CFG,
            log_scale_channels=GRAD2, diverging_cmaps=DIVERGING,
            add_coastline=False,
        )
        if im is not None:
            plt.colorbar(im, ax=ax, label=label)
    side_by_side(
        _render, LIT_DIR / f"{field}.png",
        caption=caption or ("Discussion: awaiting literature "
                            f"reference for {field}."),
    )
    plt.show()

### 5.1 density (σ₀)

Theta, Salt → rho_theta (JMD95 at p=0) → σ₀ = ρθ − 1000.  Surface-referenced potential density anomaly (`calculate_fields.potential_density_anomaly`).  Expected: ~22–28 kg m⁻³ subtropics→poles, dense outcrops in the SO.

In [ ]:
figure1_maps("density")

In [ ]:
figure2_pdfs("density")

In [ ]:
figure3_literature("density")

### 5.2 buoyancy (b)

σ₀ → b = g·σ₀/ρ₀ [m s⁻²] (`calculate_fields.buoyancy_of_field`).  Anomaly-based, so spatial structure mirrors density with sign preserved (light water = low σ₀ = low b under this convention).

In [ ]:
figure1_maps("buoyancy")

In [ ]:
figure2_pdfs("buoyancy")

In [ ]:
figure3_literature("buoyancy")

### 5.3 gradtheta2 (|∇Θ|²)

Theta → native tracer gradient (dxC, dyC) + CS/SN rotation → |∇Θ|² [°C² m⁻²] (`calculate_fields.grad_theta2`).  Fronts (Gulf Stream, ACC) should light up; halo rim is NaN.

In [ ]:
figure1_maps("gradtheta2")

In [ ]:
figure2_pdfs("gradtheta2")

In [ ]:
figure3_literature("gradtheta2")

### 5.4 gradsalt2 (|∇S|²)

Salt → |∇S|² [psu² m⁻²] (`calculate_fields.grad_salt2`).  River plumes and frontal zones dominate.

In [ ]:
figure1_maps("gradsalt2")

In [ ]:
figure2_pdfs("gradsalt2")

In [ ]:
figure3_literature("gradsalt2")

### 5.5 gradeta2 (|∇η|²)

Eta → |∇η|² [m² m⁻²] (`calculate_fields.grad_eta2`).  Proxy for geostrophic surface KE; western boundary currents and the ACC dominate.

In [ ]:
figure1_maps("gradeta2")

In [ ]:
figure2_pdfs("gradeta2")

In [ ]:
figure3_literature("gradeta2")

### 5.6 gradrho2 (|∇ρ|²)

Theta, Salt → rho_theta → |∇ρθ|² [kg² m⁻⁸] (`calculate_fields.grad_rho2`).  Density fronts; compare against gradtheta2/gradsalt2 for T- vs S-controlled regions.

In [ ]:
figure1_maps("gradrho2")

In [ ]:
figure2_pdfs("gradrho2")

In [ ]:
figure3_literature("gradrho2")

### 5.7 gradb2 (|∇b|²)

Theta, Salt → σ₀ → b → |∇b|² [s⁻⁴] (`calculate_fields.grad_b2`).  Same structure as gradrho2 scaled by (g/ρ₀)²; the frontal-structure headline field.

In [ ]:
figure1_maps("gradb2")

In [ ]:
figure2_pdfs("gradb2")

In [ ]:
figure3_literature("gradb2")

### 5.8 turner_angle (Tu)

gradtheta2, gradsalt2, gradrho2 → Tu = arctan[ρ₀(β²|∇S|² − α²|∇Θ|²)/(−|∇ρ|²/ρ₀)] [°] (`calculate_fields.turner_angle`, linear EOS α, β).  Diagnoses T- vs S-controlled density gradients; masked where |∇ρ| = 0.

In [ ]:
figure1_maps("turner_angle")

In [ ]:
figure2_pdfs("turner_angle")

In [ ]:
figure3_literature("turner_angle")

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':14s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    # Uniqueness guard: no two channels may be bit-identical.
    key = hash(sub[finite][::997].tobytes())
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:14s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

**Cross-references** — σ₀ and b are validated here; sibling notebooks
(`kinematic`, `frontogenesis`, DEPTH `stratification`, ...) reference
these sections for the shared density/buoyancy machinery rather than
re-validating it.